# Suite VIS — Visitors protocol conformance

Expressions' builders implement `Visitors.OfObject` and its parts (the protocols `FromPlain` builds through), and
expression data implements `Visitable`. Each must implement its protocol completely, with matching arity, on the class
and on live instances. (mbse-schemas' own VIS suite covers its implementations.) TypeScript checks `implements` at
compile time; these cases check at runtime.

In [ ]:
import { Expressions as E } from "@mbse/expressions";
import { assert, conformance_problems, instance_problems } from "./support.js";

## VIS-01 · Every builder, builder part and kind of data conforms to its protocol

In [ ]:
{
  const I = E._internals;
  const implementations: [{ name: string; prototype: object }, string][] = [
    [E.OfLiteral.Builder, "OfObject"], [E.OfOperation.Builder, "OfObject"], [E.OfVariable.Builder, "OfObject"],
    [E.OfLet.Builder, "OfObject"], [I._Adjacency, "OfAdjacency"], [I._Argument, "OfEntry"], [I._Link, "OfLink"],
    [I._Field, "OfProperty"], [I._Field, "OfAny"], [I._Field, "OfNative"],
    [E.OfLiteral.Data, "Visitable"], [E.OfOperation.Data, "Visitable"], [E.OfVariable.Data, "Visitable"],
    [E.OfLet.Data, "Visitable"],
  ];
  const problems = implementations.flatMap(([impl, proto]) => conformance_problems(impl, proto));
  assert(problems.length === 0, problems.join("; "));
}

## VIS-02 · Live instances conform too

In [ ]:
{
  const found: [object, string][] = [[new E.OfLiteral.Builder(), "OfObject"], [new E.OfOperation.Builder(), "OfObject"],
    [new E.OfVariable.Builder(), "OfObject"], [new E.OfLet.Builder(), "OfObject"], [E.OfLiteral.resolve(1n), "Visitable"],
    [new E.OfOperation.Data("f"), "Visitable"], [new E.OfVariable.Data("x"), "Visitable"], [new E.OfLet.Data("x"), "Visitable"]];
  const builder = new E.OfOperation.Builder();
  builder.property("name", (p: any) => { found.push([p, "OfProperty"]); p.value((a: any) => { found.push([a, "OfAny"]); a.as_native((n: any) => found.push([n, "OfNative"])); }); });
  builder.adjacency("arguments", (a: any) => { found.push([a, "OfAdjacency"]); a.add((e: any) => { found.push([e, "OfEntry"]); e.link("argument", (k: any) => found.push([k, "OfLink"])); }); });
  const problems = found.flatMap(([instance, protocol]) => instance_problems(instance, protocol));
  assert(problems.length === 0, problems.join("; "));
}